In [1]:
import sys

print(sys.executable)
print(sys.version)

c:\Users\dieve\AppData\Local\Python\pythoncore-3.11-64\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]


In [2]:
import logging
import re
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import soundfile as sf
import torch
from IPython.display import Audio, Markdown, display

warnings.filterwarnings("ignore")
logging.getLogger("chatterbox").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

SALIDAS = Path("audios")
SALIDAS.mkdir(exist_ok=True)

DISPOSITIVO = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("PyTorch:", torch.__version__)
print("Dispositivo:", DISPOSITIVO)

PyTorch: 2.6.0+cpu
Dispositivo: cpu


In [3]:
from chatterbox.mtl_tts import SUPPORTED_LANGUAGES, ChatterboxMultilingualTTS

inicio = time.time()

modelo = ChatterboxMultilingualTTS.from_pretrained(
    device=DISPOSITIVO
)

VOZ_DE_FABRICA = modelo.conds

print(f"Modelo cargado en {time.time() - inicio:.0f} segundos")
print(f"Frecuencia de audio: {modelo.sr} Hz")
print(f"Idiomas disponibles: {len(SUPPORTED_LANGUAGES)}")

Fetching 6 files: 100%|██████████| 6/6 [01:15<00:00, 12.51s/it]
Downloading: "https://github.com/explosion/spacy-pkuseg/releases/download/v0.0.26/spacy_ontonotes.zip" to C:\Users\dieve/.pkuseg\spacy_ontonotes.zip
100%|██████████| 34567143/34567143 [00:00<00:00, 35964021.90it/s]


loaded PerthNet (Implicit) at step 250,000
Modelo cargado en 143 segundos
Frecuencia de audio: 24000 Hz
Idiomas disponibles: 23


In [11]:
import sounddevice as sd

ARCHIVO_DE_VOZ = Path("mi_voz.wav")
SEGUNDOS = 15
REGRABAR = False

if ARCHIVO_DE_VOZ.exists() and not REGRABAR:
    print("Ya existe mi_voz.wav. Se utilizará esa grabación.")
else:
    microfono = sd.query_devices(kind="input")
    frecuencia = int(microfono["default_samplerate"])

    print("Micrófono:", microfono["name"])
    print("Preparando grabación...")

    for cuenta in [3, 2, 1]:
        print(cuenta)
        time.sleep(1)

    print("¡HABLA!")

    grabacion = sd.rec(
        int(SEGUNDOS * frecuencia),
        samplerate=frecuencia,
        channels=1,
        dtype="float32"
    )

    sd.wait()
    sf.write(ARCHIVO_DE_VOZ, grabacion[:, 0], frecuencia)

    print("Grabación guardada en:", ARCHIVO_DE_VOZ)

Micrófono: Micrófono de los auriculares co
Preparando grabación...
3
2
1
¡HABLA!
Grabación guardada en: mi_voz.wav


In [6]:
import librosa 

In [12]:
mi_voz, frecuencia = librosa.load(
    ARCHIVO_DE_VOZ,
    sr=None,
    mono=True
)

duracion = len(mi_voz) / frecuencia
pico = float(np.abs(mi_voz).max())
volumen = float(np.sqrt((mi_voz ** 2).mean()))

print(f"Duración: {duracion:.1f} segundos")
print(f"Volumen medio: {volumen:.3f}")
print(f"Pico máximo: {pico:.2f}")

display(Audio(mi_voz, rate=frecuencia))

Duración: 15.0 segundos
Volumen medio: 0.008
Pico máximo: 0.14


In [13]:
inicio = time.time()

texto = (
    "Hola. Esta frase no la he dicho yo: "
    "la ha generado una inteligencia artificial "
    "a partir de quince segundos de mi voz."
)

onda = modelo.generate(
    texto,
    language_id="es",
    audio_prompt_path=str(ARCHIVO_DE_VOZ),
    exaggeration=0.5,
    cfg_weight=0.5
)

primera = onda.squeeze(0).cpu().numpy()

sf.write(
    SALIDAS / "01_primera_frase.wav",
    primera,
    modelo.sr
)

print(f"Audio generado en {time.time() - inicio:.0f} segundos")

display(Audio(primera, rate=modelo.sr))

Sampling:  18%|█▊        | 180/1000 [02:08<09:43,  1.40it/s]


Audio generado en 273 segundos


In [16]:
texto_prueba = (
    "Hoy es un día fantástico para descubrir "
    "todo lo que podemos hacer con inteligencia artificial."
)

configuraciones = {
    "natural": (0.3, 0.5),
    "equilibrada": (0.5, 0.5),
    "expresiva": (0.8, 0.3)
}

for nombre, (exaggeration, cfg_weight) in configuraciones.items():

    print(f"Generando versión: {nombre}")

    inicio = time.time()

    onda = modelo.generate(
        texto_prueba,
        language_id="es",
        audio_prompt_path=str(ARCHIVO_DE_VOZ),
        exaggeration=exaggeration,
        cfg_weight=cfg_weight
    )

    audio = onda.squeeze(0).cpu().numpy()

    sf.write(
        SALIDAS / f"05_{nombre}.wav",
        audio,
        modelo.sr
    )

    print(f"Tiempo: {time.time() - inicio:.0f} segundos")

    display(Audio(audio, rate=modelo.sr))

Generando versión: natural


Sampling:  14%|█▎        | 137/1000 [01:29<09:24,  1.53it/s]


Tiempo: 189 segundos


Generando versión: equilibrada


Sampling:  16%|█▌        | 160/1000 [01:08<06:01,  2.33it/s]


Tiempo: 159 segundos


Generando versión: expresiva


Sampling:  13%|█▎        | 133/1000 [00:54<05:56,  2.43it/s]


Tiempo: 140 segundos


In [17]:
texto_largo = """
La inteligencia artificial está cambiando nuestra manera de vivir.
Hoy podemos utilizarla para estudiar, trabajar y crear contenido.
Una de sus aplicaciones más sorprendentes es la clonación de voz.
Con una grabación de tan solo quince segundos,
hemos conseguido generar frases que nunca habíamos pronunciado.
Este proyecto demuestra cómo podemos utilizar
la inteligencia artificial de una forma práctica y creativa.
"""

# Dividir el texto en frases
frases = [
    frase.strip()
    for frase in re.split(r"(?<=[.!?])\s+", texto_largo.strip())
    if frase.strip()
]

audios_generados = []

for i, frase in enumerate(frases, start=1):

    print(f"Generando fragmento {i}/{len(frases)}...")

    onda = modelo.generate(
        frase,
        language_id="es",
        audio_prompt_path=str(ARCHIVO_DE_VOZ),
        exaggeration=0.5,
        cfg_weight=0.5
    )

    audio = onda.squeeze(0).cpu().numpy()
    audios_generados.append(audio)

# Añadir pausas de medio segundo entre frases
pausa = np.zeros(int(0.5 * modelo.sr))

fragmentos = []

for i, audio in enumerate(audios_generados):
    if i > 0:
        fragmentos.append(pausa)
    fragmentos.append(audio)

audio_final = np.concatenate(fragmentos)

# Guardar el resultado
sf.write(
    SALIDAS / "06_texto_largo.wav",
    audio_final,
    modelo.sr
)

print("¡Texto largo generado correctamente!")

display(Audio(audio_final, rate=modelo.sr))

Generando fragmento 1/5...


Sampling:  10%|▉         | 99/1000 [00:39<05:58,  2.52it/s] 


Generando fragmento 2/5...


Sampling:  14%|█▍        | 145/1000 [00:50<04:56,  2.88it/s]


Generando fragmento 3/5...


...

Generando fragmento 4/5...


Sampling:  19%|█▉        | 190/1000 [01:04<04:35,  2.94it/s]


Generando fragmento 5/5...


Sampling:  17%|█▋        | 174/1000 [01:11<05:41,  2.42it/s]


¡Texto largo generado correctamente!


In [18]:
texto_ingles = (
    "Hello! My name is Diego. "
    "This is my cloned voice speaking in English. "
    "Artificial intelligence makes amazing things possible."
)

inicio = time.time()

onda = modelo.generate(
    texto_ingles,
    language_id="en",
    audio_prompt_path=str(ARCHIVO_DE_VOZ),
    exaggeration=0.5,
    cfg_weight=0.5
)

audio_ingles = onda.squeeze(0).cpu().numpy()

sf.write(
    SALIDAS / "07_voz_ingles.wav",
    audio_ingles,
    modelo.sr
)

print(f"Audio generado en {time.time() - inicio:.0f} segundos")

display(Audio(audio_ingles, rate=modelo.sr))

Sampling:  19%|█▉        | 191/1000 [01:21<05:45,  2.34it/s]


Audio generado en 188 segundos


In [19]:
import librosa
import numpy as np
from scipy.spatial.distance import cosine

# Cargar voz original y clonada
original, sr_original = librosa.load(
    "mi_voz.wav", sr=16000
)

clonada, sr_clonada = librosa.load(
    "audios/05_equilibrada.wav", sr=16000
)

# Extraer características acústicas MFCC
mfcc_original = librosa.feature.mfcc(
    y=original, sr=sr_original, n_mfcc=20
)

mfcc_clonada = librosa.feature.mfcc(
    y=clonada, sr=sr_clonada, n_mfcc=20
)

# Obtener un vector medio de cada audio
vector_original = np.mean(mfcc_original, axis=1)
vector_clonada = np.mean(mfcc_clonada, axis=1)

# Calcular similitud del coseno
similitud = 1 - cosine(vector_original, vector_clonada)

print(f"Similitud acústica: {similitud:.3f}")

Similitud acústica: 1.000


In [20]:
print(f"Similitud exacta: {similitud:.10f}")

print(f"Duración original: {len(original)/sr_original:.2f} s")
print(f"Duración clonada: {len(clonada)/sr_clonada:.2f} s")

print(f"Vectores iguales: {np.allclose(vector_original, vector_clonada)}")

Similitud exacta: 0.9995462786
Duración original: 15.00 s
Duración clonada: 6.40 s
Vectores iguales: False


In [21]:
import perth

detector = perth.PerthImplicitWatermarker()

clon_guardado, frecuencia_clon = librosa.load(SALIDAS / "01_primera_frase.wav", sr=None)

print("Marca de agua (1 = generado por el modelo, 0 = no)\n")
print("  Tu grabación original :", detector.get_watermark(mi_voz, sample_rate=frecuencia))
print("  La voz clonada        :", detector.get_watermark(clon_guardado, sample_rate=frecuencia_clon))

loaded PerthNet (Implicit) at step 250,000
Marca de agua (1 = generado por el modelo, 0 = no)

  Tu grabación original : 0.0
  La voz clonada        : 1.0
